In [0]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

# Configure App Secrets

Post-deploy task: grants App SPN READ on scope, writes embedding config.
Idempotent - safe to re-run.

In [0]:
dbutils.widgets.text("secret_scope_name", "hi_genie_credentials", "Secret Scope")
dbutils.widgets.text("app_name", "", "App Platform Name")
dbutils.widgets.text("embedding_endpoint_name", "databricks-gte-large-en", "Embedding Endpoint")
dbutils.widgets.text("embedding_endpoint_key", "embedding_endpoint_name", "Embedding Key")

scope = dbutils.widgets.get("secret_scope_name")
app_name = dbutils.widgets.get("app_name")
endpoint = dbutils.widgets.get("embedding_endpoint_name")
endpoint_key = dbutils.widgets.get("embedding_endpoint_key")

print(f"Scope: {scope} | App: {app_name} | Endpoint: {endpoint} | Key: {endpoint_key}")

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import AclPermission
w = WorkspaceClient()

## Step 1: Grant App SPN READ on scope

Looks up the App's managed SPN via `w.apps.get()`, then grants it
READ permission on the secret scope so `valueFrom` resolves at deploy.

In [0]:
if not app_name:
    print("SKIP: app_name not provided")
else:
    app_info = w.apps.get(app_name)
    spn_id = app_info.service_principal_id
    spn_details = w.service_principals.get(spn_id)
    principal = spn_details.application_id
    print(f"Resolved: {app_name} -> SPN {spn_id} -> principal {principal}")
    w.secrets.put_acl(scope=scope, principal=principal, permission=AclPermission.READ)
    print(f"  Done: scope access configured for {principal}")

## Step 2: Write embedding endpoint name to scope

Writes the Foundation Model endpoint name so the app resolves it
via `valueFrom: embedding-endpoint` in app.yaml.

In [0]:
w.secrets.put_secret(scope=scope, key=endpoint_key, string_value=endpoint)
print(f"  Done: {endpoint_key} = {endpoint}")

In [0]:
print(f"DONE. Scope '{scope}' configured for app '{app_name}'.")